# Lab 8: Trading Correlation in Prediction Markets

# Objectives

- Trade in three LMSR markets (on $A$, on $B$, and on "$A$ and $B$") and move their prices to given targets
- See that knowing how two events are related (their correlation) is information that can make money
- Compare your own profit with the class average, and see why expected profit only shows up when we pool many results


# Background

Consider two events that tend to happen together:

- $A$ = "ETH closes the week higher than it opened"
- $B$ = "BTC closes the week higher than it opened"

We run **three binary LMSR markets**: one on $A$, one on $B$, and one on "$A$ and $B$ both happen" (written A&B). As in Lab 7, a YES share pays 1 if the event happens and a NO share pays 1 if it does not. The YES price is the market's probability of the event.

If $A$ and $B$ were **independent**, we would have $P(A \text{ and } B) = P(A) \cdot P(B)$. Because ETH and BTC tend to move together, the true $P(A \text{ and } B)$ is **higher** than that.

We give you the true probabilities:

$$P(A) = 0.60, \qquad P(B) = 0.50, \qquad P(A \text{ and } B) = 0.45.$$

Every market uses liquidity $b = 100$ and starts at price $0.5$. You will play two traders, one after the other:

- **Phase 1, the independent forecaster.** Knows $P(A)$ and $P(B)$ but wrongly assumes the events are independent, so sets A&B to $0.60 \times 0.50 = 0.30$.
- **Phase 2, the correlation expert.** Knows the truth, so moves A&B up from $0.30$ to $0.45$.

At the end, your markets are resolved with **your own random draw** of what happened, using the true probabilities. Then we pool everyone's results to see which trader made money on average.

We have done all the calculations for you. Your job in class is to execute the trades, resolve your markets, and look at the results. If you want to see where the numbers come from, try the optional practice questions at the end.

## Part 1: Deploy your three markets

The contract below is a binary LMSR market, close to the one from Lab 7. There are two differences:

- Trades are paid in **play-money credits** tracked inside the contract, so you never need to send ETH, and gas does not affect your profit.
- All credit amounts and prices are reported with **18 decimals**, like wei: `1000000000000000000` means 1 credit (or a probability of 1.0). Divide by $10^{18}$ to read them.

**Instructions:**
- Create a new blank workspace in Remix and a file called `lmsr.sol`. Paste the contract below. Remix fetches the ABDK import automatically.
- Compile with Solidity **0.8.26**.
- In the Deploy tab, select **Remix VM** as the environment. Keep the same account selected for the whole lab.
- Deploy the contract **three times**, with constructor arguments `"A", 100`, then `"B", 100`, then `"A&B", 100`.
- For each deployed market, call `priceYes`. All three should return `500000000000000000` (0.5).

⚠️ **Do not refresh the Remix tab until you finish Part 5.** Your markets live in the browser's memory.

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

// Same fixed-point library as Lab 7 (Remix fetches it automatically)
import "abdk-libraries-solidity/ABDKMath64x64.sol";

/*
    Binary LMSR market for Lab 8.

    Differences from the Lab 7 contract:
      - Trades are paid in play-money "credits" that are tracked inside the contract,
        so gas fees and market funding do not pollute your PnL.
      - 1 winning share pays 1 credit. All credit amounts are reported with 18 decimals
        (1e18 = 1 credit), just like wei.
      - The cost function is written in the numerically stable "log-sum-exp" form:
            C(qYes, qNo) = b * ln(e^{qYes/b} + e^{qNo/b})
                         = max(qYes, qNo) + b * ln(1 + e^{-|qYes - qNo| / b})
        which gives the same number but never overflows.
*/
contract LMSRMarket {
    using ABDKMath64x64 for int128;

    string public name;          // "A", "B" or "A&B"
    uint256 public immutable b;  // liquidity parameter, in shares
    address public owner;

    bool public resolved;
    bool public outcome;         // true = YES won

    uint256 public qYes;         // total YES shares sold by the market maker
    uint256 public qNo;          // total NO shares sold by the market maker

    mapping(address => uint256) public yesBalance;
    mapping(address => uint256) public noBalance;
    mapping(address => uint256) public spent;   // credits paid (1e18 = 1 credit)

    event Trade(address indexed trader, bool yes, uint256 shares, uint256 cost, uint256 newPriceYes);
    event Resolved(bool outcome);

    constructor(string memory _name, uint256 _b) {
        require(_b > 0, "b must be positive");
        name = _name;
        b = _b;
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    // LMSR cost function C(y, n), in credits with 18 decimals
    function cost(uint256 y, uint256 n) public view returns (int256) {
        uint256 m = y > n ? y : n;
        uint256 d = y > n ? y - n : n - y;
        // ln(1 + e^{-d/b}) as a 64.64 fixed-point number
        int128 tail = ABDKMath64x64.divu(d, b).neg().exp().add(ABDKMath64x64.fromUInt(1)).ln();
        return int256(m) * 1e18 + tail.muli(int256(b) * 1e18);
    }

    // Credits you would pay to buy `shares` YES (yes = true) or NO (yes = false) shares
    function quoteBuy(bool yes, uint256 shares) public view returns (uint256) {
        uint256 y2 = yes ? qYes + shares : qYes;
        uint256 n2 = yes ? qNo : qNo + shares;
        return uint256(cost(y2, n2) - cost(qYes, qNo));
    }

    // Instantaneous YES price = market probability of YES (1e18 = 100%)
    function priceYes() public view returns (uint256) {
        uint256 d = qYes > qNo ? qYes - qNo : qNo - qYes;
        int128 e = ABDKMath64x64.divu(d, b).neg().exp();          // e^{-|qYes - qNo| / b}
        int128 one = ABDKMath64x64.fromUInt(1);
        int128 p = qYes >= qNo ? one.div(one.add(e)) : e.div(one.add(e));
        return p.mulu(1e18);
    }

    // Buy whole shares of one outcome. Credits are charged to your `spent` balance.
    function buy(bool yes, uint256 shares) external {
        require(!resolved, "Market resolved");
        require(shares > 0, "Zero shares");

        uint256 c = quoteBuy(yes, shares);

        if (yes) {
            qYes += shares;
            yesBalance[msg.sender] += shares;
        } else {
            qNo += shares;
            noBalance[msg.sender] += shares;
        }
        spent[msg.sender] += c;

        emit Trade(msg.sender, yes, shares, c, priceYes());
    }

    // Resolve the market (owner only). In this lab you are the owner of your own markets.
    function resolve(bool _outcome) external onlyOwner {
        require(!resolved, "Already resolved");
        resolved = true;
        outcome = _outcome;
        emit Resolved(_outcome);
    }

    // Profit and loss of `trader` after resolution, in credits with 18 decimals (can be negative)
    function pnl(address trader) external view returns (int256) {
        require(resolved, "Market not resolved");
        uint256 winning = outcome ? yesBalance[trader] : noBalance[trader];
        return int256(winning * 1e18) - int256(spent[trader]);
    }
}
```

## Part 2: Phase 1, the independent forecaster

Make these trades by calling `buy(yes, shares)` on each market. Use `true` for YES and `false` for NO. Before each trade you can call `quoteBuy(yes, shares)` to see the cost.

| Market | Call | Cost (credits) | Price after |
|---|---|---|---|
| A | `buy(true, 41)`: buy 41 YES | 22.59 | 0.601 |
| B | no trade | 0 | 0.500 |
| A&B | `buy(false, 85)`: buy 85 NO | 51.27 | 0.299 |

After trading, call `priceYes` on each market and check that it matches the table.

Why these trades? Buying YES pushes the YES price up, and buying NO pushes it down. The share counts are chosen so each price lands on its target.

## Part 3: Phase 2, the correlation expert

Make one more trade:

| Market | Call | Cost (credits) | Price after |
|---|---|---|---|
| A&B | `buy(true, 65)`: buy 65 YES | 24.23 | 0.450 |

Check `priceYes` on A&B again. Do **not** call `resolve` yet.

## Part 4: What we expect to happen

Your profit depends on which of four worlds happens. For your trades, the profit in each world is:

| World | Probability | Phase 1 profit | Phase 2 profit | Total profit |
|---|---|---|---|---|
| A and B | 0.45 | −32.86 | +40.77 | +7.91 |
| A only | 0.15 | +52.14 | −24.23 | +27.91 |
| B only | 0.05 | +11.14 | −24.23 | −13.09 |
| Neither | 0.35 | +11.14 | −24.23 | −13.09 |
| **Average (expected)** | | **−2.51** | **+5.02** | **+2.51** |

The last row is the average over the four worlds, weighted by their true probabilities.

- The **independent forecaster loses 2.51 credits on average**. By assuming independence, they pushed the A&B price down to 0.30, which is further from the truth (0.45) than where it started (0.5).
- The **correlation expert makes 5.02 credits on average**, by moving the price from 0.30 to the truth. Knowing the correlation is worth money.

But look at the Phase 2 column: in any one world, the correlation expert either wins about 41 or loses about 24. So in your own world, the "right" trader may well lose money.

## Part 5: Resolution

You get your own random draw of what happened, using the true probabilities. The draw comes from your NetID, so everyone gets a different world and anyone can check it.

Fill in your NetID and run the cell.

In [ ]:
import hashlib

NETID = "your-netid"   # lowercase, e.g. "jdoe"

# A random number in [0, 1) that depends only on your NetID
u = int(hashlib.sha256(NETID.strip().lower().encode()).hexdigest(), 16) / 2**256

# The true joint distribution: P(A&B) = 0.45, P(A & not B) = 0.15, P(not A & B) = 0.05, P(neither) = 0.35
if u < 0.45:
    A, B = True, True
elif u < 0.60:
    A, B = True, False
elif u < 0.65:
    A, B = False, True
else:
    A, B = False, False

world = {(True, True): "A and B", (True, False): "A only", (False, True): "B only", (False, False): "Neither"}[(A, B)]
print(f"your world: {world}   (u = {u:.4f})")
print(f"resolve market A   with {str(A).lower()}")
print(f"resolve market B   with {str(B).lower()}")
print(f"resolve market A&B with {str(A and B).lower()}")

Now, in Remix:

- Call `resolve` on each of your three markets with the value printed above.
- Call `pnl` with your account address on each market, and divide by $10^{18}$ to get credits.
- Check that the total matches the row of your world in the Part 4 table.

📝 Fill in the lab form now (see Submission below).

## Part 6: Class results (in class)

We pool everyone's results and put the class's average profit next to the expected values from Part 4. Things to think about:

- How many students in your world made money in Phase 2? How many lost?
- Is the class average of the Phase 2 profit close to +5.02? If not, why might that be?
- How many students (or draws) would we need for the average to reliably come out positive?

## Submission

Answer the three questions on the lab Google Form on Ed after your markets are resolved in Part 5.

**Question 1: Your world.** Select the world printed by the Part 5 cell (A and B, A only, B only, or Neither).

**Question 2: Your profit in each market.** Enter the `pnl` of markets A, B and A&B in credits, separated by commas, e.g. `18.41, 0, -10.50`.

**Question 3: Which trader makes money on average?** The independent forecaster (Phase 1), the correlation expert (Phase 2), both, or neither.

By completing this lab, you have:
- Traded in LMSR markets and moved their prices to chosen probabilities
- Seen that trading toward the truth makes money on average, and trading away from it loses money
- Seen that knowing two events are correlated is valuable information
- Seen why one person's profit says little about a strategy, while the class average does

## Optional practice questions

These questions show where all the numbers in this lab come from. They are not submitted.

**P1. The opening prices.** At opening, all three prices are $0.5$. The four worlds have probabilities $P(A \text{ and } B) = p_{AB}$, $P(A \text{ only}) = p_A - p_{AB}$, $P(B \text{ only}) = p_B - p_{AB}$ and $P(\text{neither}) = 1 - p_A - p_B + p_{AB}$. What distribution do the opening prices imply, and what does it say about $A$ and $B$?

**P2. How many shares to buy.** In a binary LMSR with liquidity $b$, the YES price is $p = 1 / \left(1 + e^{-(q_{yes} - q_{no})/b}\right)$. Show that to move the YES price from $p_0$ to $p_1$ you must buy
$$\Delta = b\left[\ln\frac{p_1}{1-p_1} - \ln\frac{p_0}{1-p_0}\right]$$
YES shares (or $-\Delta$ NO shares if $\Delta$ is negative). Check that this gives 41, 85 and 65 shares for the three trades in this lab, after rounding.

**P3. How much it costs.** Using the Lab 7 cost function $C = b \ln\left(e^{q_{yes}/b} + e^{q_{no}/b}\right)$, show that buying YES to move the price from $p_0$ to $p_1$ costs $b \ln\frac{1-p_0}{1-p_1}$. Check the costs in the Part 2 and Part 3 tables. (They differ slightly from the formula because the shares are rounded.)

**P4. Profit in each world.** A trade that buys $y$ YES shares and $n$ NO shares at cost $c$ earns $y - c$ if YES happens and $n - c$ if NO happens. Use this to rebuild the Part 4 table.

**P5. Expected profit.** If the true probability of YES is $q$, the expected profit of that trade is $y\,q + n\,(1 - q) - c$. Recompute the expected profits of Phase 1 (−2.51) and Phase 2 (+5.02).

**P6. The log scoring rule.** Show that moving the YES price from $p_0$ to $p_1$ earns $b \ln\frac{p_1}{p_0}$ if YES happens and $b \ln\frac{1-p_1}{1-p_0}$ if NO happens. If the true probability is $q$, show that the expected profit is
$$b\left[\mathrm{KL}(q \,\|\, p_0) - \mathrm{KL}(q \,\|\, p_1)\right], \qquad \mathrm{KL}(q \,\|\, p) = q\ln\frac{q}{p} + (1-q)\ln\frac{1-q}{1-p}.$$
Use this to explain why moving toward the truth always makes money on average and moving away from it loses money.

**P7. How noisy is one draw?** The Phase 2 trade wins about 40.77 with probability 0.45 and loses about 24.23 otherwise. Compute its standard deviation. With a class of 40 students, what is the standard error of the class average? About how many draws would we need for the average to be two standard errors above zero?

**P8 (Bonus). Arbitrage.** Suppose a trader pushed the A&B price to $0.55$ while the B market stays at $0.50$. Find a combination of trades in the B and A&B markets that makes money in every world.